# CUDA Kernel Optimization — SFT + GRPO RL
Fine-tuning **Qwen2.5-Coder-14B-Instruct** with QLoRA on the SakanaAI CUDA kernel archive,
followed by a GRPO reinforcement-learning stage using hardware benchmark rewards.

**Pipeline**
1. Install & authenticate
2. Shared configuration
3. Prompt templates
4. Shared utilities
5. Dataset loading & splitting
6. **Stage 1 — SFT** · warm-start (1 epoch, prompt → original kernel)
7. **Stage 2a — GRPO on level_1** · simpler kernels, higher initial reward signal
8. **Stage 2b — GRPO on level_2** · harder kernels, loaded from stage-2a checkpoint

## 1 · Installation

In [ ]:
!pip install -q unsloth
!pip install -q --upgrade --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q trl peft accelerate bitsandbytes datasets transformers scikit-learn
!pip install -q "wandb==0.19.0" "protobuf>=4.21.1,<6.0.0"

In [ ]:
! conda install -y gcc -c conda-forge

In [ ]:
!conda install -c nvidia/label/cuda-12.8.0 cuda-nvcc cuda-toolkit -y

In [15]:
!pip install ninja

In [2]:
import wandb
wandb.login()

True

In [3]:
import warnings, logging, os
warnings.filterwarnings("ignore")
logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("unsloth").setLevel(logging.ERROR)
logging.getLogger("trl").setLevel(logging.ERROR)
logging.getLogger("wandb").setLevel(logging.ERROR)
os.environ["TOKENIZERS_PARALLELISM"] = "false"

## 2 · Shared Configuration

In [4]:
import os, re, uuid, json, time, glob
import torch
import pandas as pd
from datetime import datetime
from tqdm import tqdm

# ── Model ─────────────────────────────────────────────────────────────────────
MODEL_NAME     = "unsloth/Qwen2.5-Coder-14B-Instruct"
MAX_SEQ_LENGTH = 5100
LOAD_IN_4BIT   = True

# ── LoRA ──────────────────────────────────────────────────────────────────────
LORA_R     = 16
LORA_ALPHA = 16

# ── Evaluator broker ──────────────────────────────────────────────────────────
PENDING_DIR   = "pending_kernels"
COMPLETED_DIR = "completed_results"
POLL_INTERVAL = 2.0    # seconds
POLL_TIMEOUT  = 300    # seconds

# ── Generation ────────────────────────────────────────────────────────────────
MAX_NEW_TOKENS = 2048

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42

os.makedirs(PENDING_DIR,   exist_ok=True)
os.makedirs(COMPLETED_DIR, exist_ok=True)
print("✓ Configuration loaded")

✓ Configuration loaded


## 3 · Prompt Templates
- `CUDA_OPTIMIZATION_PROMPT` — used during SFT formatting and post-training evaluation
- `CUDA_RL_PROMPT` — lean prompt used during GRPO generation to allow exploration

In [5]:
CUDA_OPTIMIZATION_PROMPT = """You are an expert in CUDA optimization. \
You will receive a CUDA kernel and must optimize it using the techniques below.

## KERNEL UNDER OPTIMIZATION
Operation : {Op_Name}

```cuda
{CUDA_Code}
```

## PRE-OPTIMIZATION CHECKLIST
Before changing anything, answer:
1. Is this kernel memory-bound or compute-bound?
2. What is the current block config and is occupancy above 50%?
3. What anti-patterns exist (uncoalesced access, bank conflicts, warp divergence)?

## OPTIMIZATION TIERS (apply in order)

Tier 1 - Thread/Block Config (10-50% gain)
- Sweep blockDim.x: 128, 256, 512; default 256
- Add __launch_bounds__(maxThreadsPerBlock, minBlocksPerMultiprocessor)
- Do not advance until occupancy > 50%

Tier 2 - Memory Access (20-40% gain)
- Tile global loads into __shared__ memory
- Pad shared arrays to avoid bank conflicts: [32][33] not [32][32]
- Use float4 for 128-bit loads; __ldg() for read-only pointers
- Ensure warp threads access consecutive addresses

Tier 3 - Tensor Cores for matmul-like ops (2-4x gain)
- wmma API: 16x16x16 (fp16) or 8x8x4 (tf32) tiles
- Accumulate in fp32; convert to fp16 only at epilogue

Tier 4 - Advanced Pipelining (10-20% gain)
- Persistent kernels: launch SM_COUNT blocks, loop over tiles inside
- cp.async (Ampere+) for async global->shared prefetch

Tier 5 - Op-Specific Patterns
- MatMul       : swizzled shmem, register-level 4x4 tiling, fused epilogue
- Softmax      : __shfl_xor_sync reduction, __expf(), online max subtraction
- LayerNorm    : Welford algorithm, __shfl_down_sync, rsqrtf()
- Cross Entropy: fused log-sum-exp, warp reductions
- RoPE         : __sincosf(), constant-memory freq table, half2 ops

## ANTI-PATTERNS TO ELIMINATE
- Warp divergence in inner loops      -> hoist branches outside
- Uncoalesced memory access           -> consecutive thread->address mapping
- Shared memory bank conflicts        -> pad or swizzle layout
- Register spilling                   -> use __launch_bounds__

## OUTPUT FORMAT
Return ONLY the optimized kernel inside a single ```cuda``` block.
The kernel must be fully self-contained and include the PYBIND11_MODULE binding.
"""

# Lean prompt for GRPO — no prescriptive tiers so the model can explore freely
CUDA_RL_PROMPT = """You are an expert CUDA kernel optimizer.

## Operation: {Op_Name}

## Original Kernel:
```cuda
{CUDA_Code}
```

Produce a single optimized CUDA kernel that is faster than the original while \
preserving correctness. Output only the optimized kernel inside a ```cuda``` block. \
The kernel must be fully self-contained and include the PYBIND11_MODULE binding.
"""

print("✓ Prompt templates defined")

✓ Prompt templates defined


## 4 · Shared Utilities

In [6]:
# ── 4.1  Code extraction ──────────────────────────────────────────────────────
def extract_cuda_code(response: str) -> str | None:
    """Extract the first valid CUDA kernel from a model response."""
    # Pass 1: tagged fence (```cuda / ```cpp / ```c++) with CUDA keywords
    pattern = r"```(?:cuda|cpp|c\+\+)?\s*([\s\S]*?)```"
    for m in re.findall(pattern, response, re.IGNORECASE):
        if any(kw in m for kw in ["__global__", "cudaMalloc", "blockIdx",
                                   "threadIdx", "PYBIND11_MODULE"]):
            return m.strip()
    # Pass 2: any fenced block containing CUDA keywords (untagged fences)
    for m in re.findall(r"```\w*\n?([\s\S]*?)```", response):
        if any(kw in m for kw in ["__global__", "blockIdx", "threadIdx"]):
            return m.strip()
    # Pass 3: no fences at all — model dumped raw code
    if "__global__" in response:
        return response.strip()
    return None


# ── 4.2  Kernel generation ────────────────────────────────────────────────────
def generate_kernel(model, tokenizer, op_name: str, cuda_code: str,
                    prompt_template: str = CUDA_OPTIMIZATION_PROMPT,
                    max_new_tokens: int = MAX_NEW_TOKENS) -> str:
    """Run one forward pass and return the raw model response string."""
    from unsloth import FastLanguageModel
    FastLanguageModel.for_inference(model)

    if not isinstance(cuda_code, str) or not cuda_code.strip():
        return ""
    if not isinstance(op_name, str) or not op_name.strip():
        op_name = "unknown_op"

    code_tokens = tokenizer.encode(cuda_code, add_special_tokens=False)
    if len(code_tokens) > 1200:
        cuda_code = tokenizer.decode(code_tokens[:1200]) + "\n// ... (truncated)"

    messages = [
        {"role": "system",
         "content": prompt_template.format(Op_Name=op_name, CUDA_Code=cuda_code)},
        {"role": "user",
         "content": (f"Optimize this CUDA kernel for operation: {op_name}\n\n"
                      f"```cuda\n{cuda_code}\n```")},
    ]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.2,
            top_p=0.95,
            repetition_penalty=1.1,
            use_cache=True,
        )
    new_ids = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_ids, skip_special_tokens=True)


# ── 4.3  Evaluator broker ─────────────────────────────────────────────────────
def evaluate_kernel_pair(candidate_code: str, reference_code: str,
                         op_name: str) -> dict:
    """
    Submit a (candidate, reference) pair to the async evaluator via the
    file broker. Also writes a _meta.json sidecar so the evaluator knows
    the op_name and can generate the right input tensors.
    correctness = (status == 'SUCCESS') — no dependency on metrics fields.
    """
    trial_id    = f"{op_name}_{uuid.uuid4().hex[:8]}"
    cand_path   = os.path.join(PENDING_DIR,   f"{trial_id}_candidate.cu")
    ref_path    = os.path.join(PENDING_DIR,   f"{trial_id}_reference.cu")
    meta_path   = os.path.join(PENDING_DIR,   f"{trial_id}_meta.json")
    result_path = os.path.join(COMPLETED_DIR, f"{trial_id}_results.json")

    os.makedirs(PENDING_DIR,   exist_ok=True)
    os.makedirs(COMPLETED_DIR, exist_ok=True)

    with open(cand_path, "w") as f: f.write(candidate_code)
    with open(ref_path,  "w") as f: f.write(reference_code)
    with open(meta_path, "w") as f: json.dump({"op_name": op_name}, f)

    deadline = time.time() + POLL_TIMEOUT
    while time.time() < deadline:
        if os.path.exists(result_path):
            with open(result_path) as f:
                result = json.load(f)
            status        = result.get("status", "")
            executability = status not in ("COMPILATION_ERROR", "RUNTIME_ERROR",
                                           "POLL_TIMEOUT", "SKIP_MISSING_KERNEL")
            correctness   = (status == "SUCCESS")
            return {
                "trial_id":      trial_id,
                "op_name":       op_name,
                "status":        status,
                "executability": executability,
                "correctness":   correctness,
                "reward":        result.get("reward", -1.0),
                "speedup":       result.get("speedup"),
                "metrics":       result.get("metrics", {}),
                "error_trace":   result.get("error_trace"),
            }
        time.sleep(POLL_INTERVAL)

    return {"trial_id": trial_id, "op_name": op_name, "status": "POLL_TIMEOUT",
            "executability": False, "correctness": False,
            "reward": -1.0, "speedup": None}


# ── 4.4  Batch evaluation ─────────────────────────────────────────────────────
def run_evaluation(model, tokenizer, rows: list[dict],
                   prompt_template: str = CUDA_OPTIMIZATION_PROMPT,
                   tag: str = "") -> list[dict]:
    """Evaluate the model on a list of dataset rows and return result dicts."""
    records = []
    for row in tqdm(rows, desc=f"Evaluating [{tag}]"):
        op_name   = row.get("Op_Name")   or "unknown_op"
        cuda_code = row.get("CUDA_Code") or ""

        if not cuda_code.strip():
            records.append({"op_name": op_name, "executability": False,
                            "correctness": False, "reward": -1.0,
                            "error": "empty CUDA_Code"})
            continue

        response  = generate_kernel(model, tokenizer, op_name, cuda_code,
                                    prompt_template=prompt_template)
        extracted = extract_cuda_code(response)

        if extracted is None:
            records.append({"op_name": op_name, "executability": False,
                            "correctness": False, "reward": -1.0,
                            "error": "no CUDA block in response"})
            print(f"  [WARN] extract_cuda_code returned None for op={op_name}. Response preview: {response[:200]!r}")
            continue

        result = evaluate_kernel_pair(
            candidate_code=extracted, reference_code=cuda_code, op_name=op_name)
        records.append(result)
    return records


# ── 4.5  Summary printer ──────────────────────────────────────────────────────
def print_summary(results_df: pd.DataFrame, label: str = "EVALUATION") -> dict:
    """Print and return summary metrics from a results DataFrame."""
    exec_rate    = results_df["executability"].mean() if "executability" in results_df else float("nan")
    correct_rate = results_df["correctness"].mean()  if "correctness"   in results_df else float("nan")
    mean_reward  = results_df["reward"].mean()        if "reward"        in results_df else float("nan")
    mean_speedup = (results_df["speedup"].dropna().mean()
                    if "speedup" in results_df and results_df["speedup"].notna().any()
                    else float("nan"))

    print(f"\n{'='*60}")
    print(f"{label}")
    print(f"{'='*60}")
    print(f"  Samples            : {len(results_df)}")
    print(f"  Executability rate : {exec_rate:.2%}")
    print(f"  Correctness rate   : {correct_rate:.2%}")
    print(f"  Mean reward        : {mean_reward:.4f}")
    print(f"  Mean speedup       : {mean_speedup:.4f}x")

    return {"exec_rate": exec_rate, "correct_rate": correct_rate,
            "mean_reward": mean_reward, "mean_speedup": mean_speedup}

print("✓ Shared utilities defined")

✓ Shared utilities defined


## 5 · Dataset Loading & Splitting

| Split | Used for | Holdout |
|---|---|---|
| level_1 | SFT warm-start + GRPO stage-2a | 1% test |
| level_2 | GRPO stage-2b | 5% test |

All splits are done once here to avoid any leakage.

In [7]:
from datasets import load_dataset
from sklearn.model_selection import train_test_split

def load_and_filter(split: str) -> pd.DataFrame:
    raw = load_dataset("SakanaAI/AI-CUDA-Engineer-Archive", split=split)
    df  = raw.to_pandas()
    df  = df[
        (df["Correct"] == True) &
        df["CUDA_Code"].notna() &
        (df["CUDA_Code"].str.strip() != "")
    ].reset_index(drop=True)
    print(f"  {split}: {len(df):,} correct kernels after filtering")
    return df

print("Loading datasets …")
l1_df = load_and_filter("level_1")
# l2_df = load_and_filter("level_2")

# level_1 — 1% test holdout; rest used for both SFT and GRPO stage-2a
l1_train_df, l1_test_df = train_test_split(l1_df, test_size=0.01, random_state=SEED)
l1_train_df = l1_train_df.reset_index(drop=True)
l1_test_df  = l1_test_df.reset_index(drop=True)

# level_2 — 5% test holdout; rest used for GRPO stage-2b
# l2_train_df, l2_test_df = train_test_split(l2_df, test_size=0.05, random_state=SEED)
# l2_train_df = l2_train_df.reset_index(drop=True)
# l2_test_df  = l2_test_df.reset_index(drop=True)

print(f"\nlevel_1 — train: {len(l1_train_df):,}  |  test: {len(l1_test_df):,}")
# print(f"level_2 — train: {len(l2_train_df):,}  |  test: {len(l2_test_df):,}")

Loading datasets …


  level_1: 7,222 correct kernels after filtering

level_1 — train: 7,149  |  test: 73


## 6 · Stage 1 — SFT Warm-Start
**Goal:** teach the model the output format and PYBIND11 structure, not optimization.
- 1 epoch only (warm-start, not deep fine-tuning)
- Target = original kernel (prompt → original kernel)
- level_1 training split

### 6.1 · Load model & attach LoRA

In [8]:
from unsloth import FastLanguageModel, is_bfloat16_supported

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name    = MODEL_NAME,
    max_seq_length= MAX_SEQ_LENGTH,
    dtype         = None,
    load_in_4bit  = LOAD_IN_4BIT,
)

model = FastLanguageModel.get_peft_model(
    model,
    r                         = LORA_R,
    target_modules            = ["q_proj", "k_proj", "v_proj", "o_proj",
                                 "gate_proj", "up_proj", "down_proj"],
    lora_alpha                = LORA_ALPHA,
    lora_dropout              = 0,
    bias                      = "none",
    use_gradient_checkpointing= "unsloth",
    random_state              = SEED,
    use_rslora                = False,
)
model.print_trainable_parameters()

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.8: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA H100 NVL MIG 1g.24gb. Num GPUs = 1. Max memory: 21.625 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 9.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/579 [00:00<?, ?it/s]

Unsloth 2026.4.8 patched 48 layers with 48 QKV layers, 48 O layers and 48 MLP layers.


trainable params: 68,812,800 || all params: 14,838,846,464 || trainable%: 0.4637


### 6.2 · Format SFT dataset

In [9]:
from datasets import Dataset

EOS_TOKEN = tokenizer.eos_token

def format_sft_row(examples):
    """
    Warm-start format: assistant target = original kernel.
    Teaches format and PYBIND11 structure, not optimization.
    """
    texts = []
    for op, cuda_code in zip(examples["Op_Name"], examples["CUDA_Code"]):
        tokens = tokenizer.encode(cuda_code, add_special_tokens=False)
        if len(tokens) > 1200:
            cuda_code = tokenizer.decode(tokens[:1200]) + "\n// ... (truncated)"

        messages = [
            {"role": "system",
             "content": CUDA_OPTIMIZATION_PROMPT.format(Op_Name=op, CUDA_Code=cuda_code)},
            {"role": "user",
             "content": (f"Optimize this CUDA kernel for operation: {op}\n\n"
                          f"```cuda\n{cuda_code}\n```")},
            {"role": "assistant",
             "content": f"```cuda\n{cuda_code}\n```"},  # target = original kernel (warm-start)
        ]
        text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False)
        texts.append(text + EOS_TOKEN)
    return {"text": texts}

sft_hf = Dataset.from_pandas(l1_train_df)
sft_hf = sft_hf.map(format_sft_row, batched=True, remove_columns=sft_hf.column_names)
print(f"SFT dataset: {len(sft_hf):,} rows")
print("Preview:\n", sft_hf[0]["text"][:400], "...")

Map:   0%|          | 0/7149 [00:00<?, ? examples/s]

SFT dataset: 7,149 rows
Preview:
 <|im_start|>system
You are an expert in CUDA optimization. You will receive a CUDA kernel and must optimize it using the techniques below.

## KERNEL UNDER OPTIMIZATION
Operation : 89_cumsum

```cuda
#include <torch/extension.h>
#include <cuda.h>
#include <cuda_runtime.h>

#define CHECK_CUDA(x) TORCH_CHECK(x.is_cuda(), #x " must be a CUDA tensor")
#define CHECK_CONTIGUOUS(x) TORCH_CHECK(x.is_conti ...


### 6.3 · WandB init & reward callback

In [11]:
import wandb
from transformers import TrainerCallback

wandb.init(
    project = "cuda-kernel-optimization",
    name    = "cuda_14b_sft_warmstart",
    notes   = "SFT warm-start — 1 epoch, prompt→original kernel, level_1",
)

class RewardLoggingCallback(TrainerCallback):
    """Periodically probe the model on a small held-out set during SFT."""

    def __init__(self, model, tokenizer, probe_rows: list[dict], every_n_steps: int = 100):
        self.model         = model
        self.tokenizer     = tokenizer
        self.probe_rows    = probe_rows
        self.every_n_steps = every_n_steps

    def on_step_end(self, args, state, control, **kwargs):
        if state.global_step % self.every_n_steps != 0:
            return

        records      = run_evaluation(self.model, self.tokenizer, self.probe_rows, tag="sft-probe")
        rewards      = [r.get("reward", -1.0)       for r in records]
        exec_flags   = [r.get("executability", False) for r in records]
        correct_flags= [r.get("correctness",   False) for r in records]

        mean_reward  = sum(rewards)        / len(rewards)        if rewards else -1.0
        exec_rate    = sum(exec_flags)     / len(exec_flags)     if exec_flags else 0.0
        correct_rate = sum(correct_flags)  / len(correct_flags)  if correct_flags else 0.0

        wandb.log({"sft/mean_reward": mean_reward, "sft/exec_rate": exec_rate,
                   "sft/correct_rate": correct_rate, "sft/step": state.global_step})
        print(f"[SFT step {state.global_step}] reward={mean_reward:.3f} "
              f"exec={exec_rate:.2%} correct={correct_rate:.2%}")

print("✓ WandB run started & callback defined")

✓ WandB run started & callback defined


### 6.4 · Train

In [23]:
from trl import SFTTrainer
from transformers import TrainingArguments

SFT_CHECKPOINT_DIR = "outputs/stage1_sft_v2"

def get_latest_checkpoint(output_dir):
    checkpoints = glob.glob(os.path.join(output_dir, "checkpoint-*"))
    if not checkpoints:
        return None
    latest = max(checkpoints, key=lambda p: int(p.split("-")[-1]))
    print(f"Resuming from: {latest}")
    return latest

resume_from = get_latest_checkpoint(SFT_CHECKPOINT_DIR)

probe_rows = l1_test_df.head(5).to_dict(orient="records")
callback   = RewardLoggingCallback(model, tokenizer, probe_rows, every_n_steps=200)

trainer = SFTTrainer(
    model             = model,
    tokenizer         = tokenizer,
    train_dataset     = sft_hf,
    dataset_text_field= "text",
    max_seq_length    = MAX_SEQ_LENGTH,
    dataset_num_proc  = 2,
    packing           = False,
    args = TrainingArguments(
        num_train_epochs             = 1,       # warm-start only
        per_device_train_batch_size  = 1,
        gradient_accumulation_steps  = 4,
        warmup_ratio                 = 0.05,
        learning_rate                = 5e-5,
        lr_scheduler_type            = "cosine",
        fp16                         = not is_bfloat16_supported(),
        bf16                         = is_bfloat16_supported(),
        logging_steps                = 10,
        save_steps                   = 100,
        save_total_limit             = 3,
        optim                        = "adamw_8bit",
        weight_decay                = 0.01,
        max_grad_norm                = 1.0,
        seed                         = SEED,
        report_to                    = "wandb",
        output_dir                   = SFT_CHECKPOINT_DIR,
    ),
)
trainer.add_callback(callback)
trainer.train(resume_from_checkpoint=resume_from)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Resuming from: outputs/stage1_sft_v2/checkpoint-800


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Tokenizing ["text"] (num_proc=64):   0%|          | 0/7149 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 7,149 | Num Epochs = 1 | Total steps = 1,788
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 68,812,800 of 14,838,846,464 (0.46% trained)


...

[SFT step 1400] reward=-1.000 exec=0.00% correct=0.00%


Unsloth: Restored added_tokens_decoder metadata in outputs/stage1_sft_v2/checkpoint-1400/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/stage1_sft_v2/checkpoint-1500/tokenizer_config.json.
Both `max_new_tokens` (=2048) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=2048) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=2048) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tok

[SFT step 1600] reward=-1.000 exec=0.00% correct=0.00%


Unsloth: Restored added_tokens_decoder metadata in outputs/stage1_sft_v2/checkpoint-1600/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/stage1_sft_v2/checkpoint-1700/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/stage1_sft_v2/checkpoint-1788/tokenizer_config.json.


TrainOutput(global_step=1788, training_loss=0.10070651989655206, metrics={'train_runtime': 10409.0837, 'train_samples_per_second': 0.687, 'train_steps_per_second': 0.172, 'total_flos': 6.175760365167575e+17, 'train_loss': 0.10070651989655206, 'epoch': 1.0})

### 6.5 · Save SFT checkpoint

In [24]:
model.save_pretrained("outputs/stage1_sft_v2_final")
tokenizer.save_pretrained("outputs/stage1_sft_v2_final")
print("✓ SFT checkpoint saved → outputs/stage1_sft_v2_final")

Unsloth: Restored added_tokens_decoder metadata in outputs/stage1_sft_v2_final/tokenizer_config.json.


✓ SFT checkpoint saved → outputs/stage1_sft_v2_final


### 6.6 · Post-SFT evaluation

In [26]:
from unsloth import FastLanguageModel

FastLanguageModel.for_inference(model)

sft_results    = run_evaluation(model, tokenizer, l1_test_df.to_dict(orient="records"), tag="post-sft")
sft_results_df = pd.DataFrame(sft_results)

metrics = print_summary(sft_results_df, "POST-SFT EVALUATION")
wandb.log({f"post_sft/{k}": v for k, v in metrics.items()})

sft_results_df.to_json("post_sft_results.json", orient="records", indent=2)
print("✓ Results saved → post_sft_results.json")
wandb.finish()

Both `max_new_tokens` (=2048) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Evaluating [post-sft]:   0%|          | 0/73 [01:53<?, ?it/s]


KeyboardInterrupt: 

## 7 · Stage 2 — GRPO Reinforcement Learning

Two sequential GRPO stages:
- **Stage 2a** — trained on level_1 (simpler kernels, higher chance of initial correct generations)
- **Stage 2b** — loaded from stage-2a checkpoint, trained on level_2 (harder kernels)

The reward function and dataset builder are shared between both stages.

### 7.1 · Shared reward function & dataset builder

In [ ]:
def _benchmark_reward(candidate_code: str, reference_code: str, op_name: str) -> float:
    """Submit to evaluator broker via file protocol and return reward scalar."""
    trial_id    = f"rl_{op_name}_{uuid.uuid4().hex[:8]}"
    cand_path   = os.path.join(PENDING_DIR,   f"{trial_id}_candidate.cu")
    ref_path    = os.path.join(PENDING_DIR,   f"{trial_id}_reference.cu")
    meta_path   = os.path.join(PENDING_DIR,   f"{trial_id}_meta.json")
    result_path = os.path.join(COMPLETED_DIR, f"{trial_id}_results.json")

    with open(cand_path, "w") as f: f.write(candidate_code)
    with open(ref_path,  "w") as f: f.write(reference_code)
    with open(meta_path, "w") as f: json.dump({"op_name": op_name}, f)

    deadline = time.time() + POLL_TIMEOUT
    while time.time() < deadline:
        if os.path.exists(result_path):
            with open(result_path) as f:
                result = json.load(f)
            return float(result.get("reward", -1.0))
        time.sleep(POLL_INTERVAL)
    return -1.0


def grpo_reward_fn(completions, reference_kernel, op_name, **kwargs) -> list[float]:
    """Reward function passed to GRPOTrainer. Parameter names match dataset columns."""
    rewards = []
    for sample_idx, sample_completions in enumerate(completions):
        ref = reference_kernel[sample_idx]
        op  = op_name[sample_idx]
        for gen in sample_completions:
            content   = gen["content"] if isinstance(gen, dict) else gen
            extracted = extract_cuda_code(content)
            reward    = _benchmark_reward(extracted, ref, op) if extracted else -1.0
            rewards.append(reward)
    return rewards


from datasets import Dataset as HFDataset

def build_grpo_dataset(df: pd.DataFrame, tokenizer,
                        max_prompt_tokens: int = 1800) -> HFDataset:
    """Wrap each row into the schema expected by GRPOTrainer."""
    rows = []
    for _, row in df.iterrows():
        op_name   = row["Op_Name"]   or "unknown_op"
        cuda_code = row["CUDA_Code"] or ""
        if not cuda_code.strip():
            continue
        tokens = tokenizer.encode(cuda_code, add_special_tokens=False)
        if len(tokens) > max_prompt_tokens:
            cuda_code = tokenizer.decode(tokens[:max_prompt_tokens]) + "\n// ... (truncated)"
        prompt = [
            {"role": "system",
             "content": CUDA_RL_PROMPT.format(Op_Name=op_name, CUDA_Code=cuda_code)},
            {"role": "user",
             "content": (f"Optimize this CUDA kernel for operation: {op_name}\n\n"
                          f"```cuda\n{cuda_code}\n```")},
        ]
        rows.append({"prompt": prompt, "reference_kernel": cuda_code, "op_name": op_name})
    return HFDataset.from_list(rows)


def make_grpo_config(output_dir: str, wandb_name: str, wandb_notes: str) -> tuple:
    """Return (GRPOConfig, wandb_run). Centralises all GRPO hyper-parameters."""
    from trl import GRPOConfig
    wandb.init(project="cuda-kernel-optimization", name=wandb_name, notes=wandb_notes)
    cfg = GRPOConfig(
        num_generations             = 4,
        max_new_tokens              = 1024,
        temperature                 = 0.8,
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 8,
        num_train_epochs            = 1,
        learning_rate               = 5e-6,
        lr_scheduler_type           = "cosine",
        warmup_ratio                = 0.05,
        optim                       = "adamw_8bit",
        weight_decay                = 0.01,
        max_grad_norm               = 0.1,
        beta                        = 0.01,
        epsilon                     = 0.2,
        loss_type                   = "grpo",
        bf16                        = True,
        fp16                        = False,
        max_prompt_length           = 4000,
        max_completion_length       = 1024,
        logging_steps               = 5,
        save_steps                  = 50,
        save_total_limit            = 3,
        output_dir                  = output_dir,
        report_to                   = "wandb",
        seed                        = SEED,
    )
    return cfg

print("✓ GRPO shared helpers defined")

### 7.2 · Stage 2a — GRPO on level_1
Load the SFT checkpoint, run GRPO on level_1 (simpler kernels).
level_1 kernels are more likely to produce correct generations early, giving the reward signal something to work with.

In [ ]:
from unsloth import FastLanguageModel
from trl import GRPOTrainer

# Load SFT checkpoint
rl_model, rl_tokenizer = FastLanguageModel.from_pretrained(
    model_name    = "outputs/stage1_sft_final",
    max_seq_length= MAX_SEQ_LENGTH,
    dtype         = None,
    load_in_4bit  = LOAD_IN_4BIT,
)
rl_model = FastLanguageModel.get_peft_model(
    rl_model,
    r=LORA_R, target_modules=["q_proj","k_proj","v_proj","o_proj",
                               "gate_proj","up_proj","down_proj"],
    lora_alpha=LORA_ALPHA, lora_dropout=0, bias="none",
    use_gradient_checkpointing="unsloth", random_state=SEED, use_rslora=False,
)
rl_model.print_trainable_parameters()

# Build dataset from level_1 train split
l1_grpo_df      = l1_train_df.sample(n=min(300, len(l1_train_df)), random_state=SEED).reset_index(drop=True)
grpo_dataset_l1 = build_grpo_dataset(l1_grpo_df, rl_tokenizer)
print(f"Stage-2a dataset: {len(grpo_dataset_l1)} samples (level_1)")

grpo_cfg_l1 = make_grpo_config(
    output_dir  = "outputs/stage2a_grpo_l1",
    wandb_name  = "cuda_14b_grpo_l1",
    wandb_notes = "GRPO stage-2a — level_1, loaded from SFT warm-start",
)

grpo_trainer_l1 = GRPOTrainer(
    model            = rl_model,
    processing_class = rl_tokenizer,
    config           = grpo_cfg_l1,
    train_dataset    = grpo_dataset_l1,
    reward_funcs     = grpo_reward_fn,
)
print("✓ Starting GRPO stage-2a (level_1) …")
grpo_trainer_l1.train()

In [ ]:
rl_model.save_pretrained("outputs/stage2a_grpo_l1_final")
rl_tokenizer.save_pretrained("outputs/stage2a_grpo_l1_final")
print("✓ Stage-2a checkpoint saved → outputs/stage2a_grpo_l1_final")

##### Post stage-2a evaluation (level_1 test split)

In [ ]:
from unsloth import FastLanguageModel

FastLanguageModel.for_inference(rl_model)

l1_rl_results    = run_evaluation(rl_model, rl_tokenizer,
                                   l1_test_df.to_dict(orient="records"),
                                   prompt_template=CUDA_RL_PROMPT, tag="post-grpo-l1")
l1_rl_results_df = pd.DataFrame(l1_rl_results)

metrics = print_summary(l1_rl_results_df, "POST GRPO STAGE-2a (level_1)")
wandb.log({f"post_grpo_l1/{k}": v for k, v in metrics.items()})

l1_rl_results_df.to_json("post_grpo_l1_results.json", orient="records", indent=2)
print("✓ Results saved → post_grpo_l1_results.json")
wandb.finish()

### 7.3 · Stage 2b — GRPO on level_2
Load the stage-2a checkpoint, run GRPO on level_2 (harder kernels, unseen during SFT).

In [ ]:
from unsloth import FastLanguageModel
from trl import GRPOTrainer

# Load stage-2a checkpoint
rl_model2, rl_tokenizer2 = FastLanguageModel.from_pretrained(
    model_name    = "outputs/stage2a_grpo_l1_final",
    max_seq_length= MAX_SEQ_LENGTH,
    dtype         = None,
    load_in_4bit  = LOAD_IN_4BIT,
)
rl_model2 = FastLanguageModel.get_peft_model(
    rl_model2,
    r=LORA_R, target_modules=["q_proj","k_proj","v_proj","o_proj",
                               "gate_proj","up_proj","down_proj"],
    lora_alpha=LORA_ALPHA, lora_dropout=0, bias="none",
    use_gradient_checkpointing="unsloth", random_state=SEED, use_rslora=False,
)
rl_model2.print_trainable_parameters()

# Build dataset from level_2 train split
l2_grpo_df      = l2_train_df.sample(n=min(300, len(l2_train_df)), random_state=SEED).reset_index(drop=True)
grpo_dataset_l2 = build_grpo_dataset(l2_grpo_df, rl_tokenizer2)
print(f"Stage-2b dataset: {len(grpo_dataset_l2)} samples (level_2)")

grpo_cfg_l2 = make_grpo_config(
    output_dir  = "outputs/stage2b_grpo_l2",
    wandb_name  = "cuda_14b_grpo_l2",
    wandb_notes = "GRPO stage-2b — level_2, loaded from stage-2a checkpoint",
)

grpo_trainer_l2 = GRPOTrainer(
    model            = rl_model2,
    processing_class = rl_tokenizer2,
    config           = grpo_cfg_l2,
    train_dataset    = grpo_dataset_l2,
    reward_funcs     = grpo_reward_fn,
)
print("✓ Starting GRPO stage-2b (level_2) …")
grpo_trainer_l2.train()

In [ ]:
rl_model2.save_pretrained("outputs/stage2b_grpo_l2_final")
rl_tokenizer2.save_pretrained("outputs/stage2b_grpo_l2_final")
print("✓ Final checkpoint saved → outputs/stage2b_grpo_l2_final")

##### Post stage-2b evaluation (level_2 test split)

In [ ]:
from unsloth import FastLanguageModel

FastLanguageModel.for_inference(rl_model2)

l2_rl_results    = run_evaluation(rl_model2, rl_tokenizer2,
                                   l2_test_df.to_dict(orient="records"),
                                   prompt_template=CUDA_RL_PROMPT, tag="post-grpo-l2")
l2_rl_results_df = pd.DataFrame(l2_rl_results)

metrics = print_summary(l2_rl_results_df, "POST GRPO STAGE-2b (level_2)")
wandb.log({f"post_grpo_l2/{k}": v for k, v in metrics.items()})

l2_rl_results_df.to_json("post_grpo_l2_results.json", orient="records", indent=2)
print("✓ Results saved → post_grpo_l2_results.json")
wandb.finish()